<div align="center">
  <a href="https://colab.research.google.com/github/PrunaAI/ai-efficiency-courses/blob/main/solutions/01-analyze_llm_architectures.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
  </a>
</div>

---
**💡 Tip**: Click the button above to open this notebook in Google Colab for free GPU access!

## Installation

This notebook includes automatic setup cells that will install the project from git repository with UV.

**Note**: Run the setup cells below before starting the exercises.

In [ ]:
# Install project directly from git repository
!uv pip install git+https://github.com/PrunaAI/ai-efficiency-courses.git

## Utility cells

During the course, we'll leverage some course utilities to streamline our workflow. These utilities are located in the `course` package, which can simply be imported given that we installed the project from git repository above.
You can find the source code [here](https://github.com/PrunaAI/ai-efficiency-courses/tree/main/course).

These utilities will help us:
- Load and manage lists of model ids that we have verified to work.
- Generate informative plots for model analysis.
- Iterate efficiently over evaluation and model configuration options.

Let's first load our models. We will use `SMALL_MODEL_IDS`, which are sub 1B parameters which should be easy to download and load into memory. We recommend starting with these smaller models but feel free to experiment with other models until you reach your GPU memory limit!

In [ ]:
from course import SMALL_MODEL_IDS, MEDIUM_MODEL_IDS, LARGE_MODEL_IDS, ALL_MODEL_IDS

MODEL_IDS = SMALL_MODEL_IDS
# MODEL_IDS = MEDIUM_MODEL_IDS
# MODEL_IDS = LARGE_MODEL_IDS
# MODEL_IDS = ALL_MODEL_IDS

MODEL_IDS

We also recommend to set a custom cache directory for models. Loading models can take significant disk space. To avoid filling up your default disk, we recommend setting a custom cache directory for downloaded models. You can do this by running the following in a terminal or in a notebook cell:

In [ ]:
# Replace <path_to_cache> with your desired cache path
import os

CACHE_PATH = "<path_to_cache>"
os.environ["TORCH_HOME"] = CACHE_PATH
os.environ["HF_HOME"] = CACHE_PATH
os.environ["HUGGINGFACE_HUB_CACHE"] = CACHE_PATH
os.environ["HUGGINGFACE_ASSETS_CACHE"] = CACHE_PATH

You can also clear the cache by running the following cell:

In [ ]:
from course.models import clear_cache

clear_cache(CACHE_PATH)

# 01: Analyze LLM Architectures

Welcome to the first unit of the AI Efficiency course! 🚀

In this tutorial, we will take a deep dive into the internal architecture of Large Language Models (LLMs). Understanding the building blocks of LLMs is crucial for anyone working with these models—whether you want to debug, optimize, or simply gain a deeper appreciation for how they work. The content from the chapter 1 [slides](https://github.com/PrunaAI/ai-efficiency-courses/blob/main/slides/01-language_model_architectures.pdf) will help you to go through this notebook.


By the end of this unit, you will:
- Understand the key components that make up modern LLMs.
- Learn how to inspect and analyze model architectures programmatically.
- Gain practical skills for evaluating model capabilities and limitations.

Let's get started on unraveling the inner workings of LLMs!

## 1. Imports

As we've already installed the project, we can import the necessary libraries. We will be using `torch` and `transformers` for this tutorial as interfaces to the model and tokenizer. On top of that, we will be using `matplotlib` and `numpy` for basic plotting.

In [5]:
import matplotlib.pyplot as plt
import numpy as np
import torch

from transformers import AutoModelForCausalLM, AutoTokenizer

Beyond external libraries, this course comes with the `course` local package which contains a lot of utils that you can use in the notebooks. We will be using `create_single_plot` to create a simple plot.

In [6]:
from course import create_single_plot

## 2. Analyze LLM Architectures

### 2.1 Explore General LLM Model Information

In this section, you'll learn how to systematically extract and interpret the core architectural properties of large language models (LLMs).

**Why is this important?**
Understanding the structure of an LLM—such as its hidden size, number of layers, and vocabulary size—helps you reason about its capabilities, memory requirements, and potential performance. These attributes are foundational for comparing models and making informed choices for downstream tasks.

**Your tasks:**
1. Implement the `get_model_info` function that takes a loaded LLM model and returns a dictionary with its key architectural details.
2. Apply your function to multiple models to extract and compare information from several LLMs. This will help you see how different design choices manifest across models.

**What to think about:**
- What is the hidden size of the model? (This determines the dimensionality of the model's internal representations.)
- How many attention heads does it use? (Each head compute different attention between input tokens.)
- How many layers and blocks are there? (Deeper models can often capture more complex patterns.)
- What is the vocabulary size? (Elements of the vocabulary are tokens that the model can represent directly.)
- What is the architecture class? (E.g., "OPTForCausalLM", "LlamaForCausalLM", etc.)
- What is the maximum context length? (How many tokens can the model process at once?)

As you complete this section, reflect on how these properties might influence the model's strengths, weaknesses, and resource requirements.

In [7]:
# Select the model to analyze
model_id = MODEL_IDS[0]

# Load the model and tokenizer
model = AutoModelForCausalLM.from_pretrained(model_id)
tokenizer = AutoTokenizer.from_pretrained(model_id)
model, tokenizer

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/441 [00:00<?, ?B/s]

(OPTForCausalLM(
   (model): OPTModel(
     (decoder): OPTDecoder(
       (embed_tokens): Embedding(50272, 768, padding_idx=1)
       (embed_positions): OPTLearnedPositionalEmbedding(2050, 768)
       (final_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
       (layers): ModuleList(
         (0-11): 12 x OPTDecoderLayer(
           (self_attn): OPTAttention(
             (k_proj): Linear(in_features=768, out_features=768, bias=True)
             (v_proj): Linear(in_features=768, out_features=768, bias=True)
             (q_proj): Linear(in_features=768, out_features=768, bias=True)
             (out_proj): Linear(in_features=768, out_features=768, bias=True)
           )
           (activation_fn): ReLU()
           (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
           (fc1): Linear(in_features=768, out_features=3072, bias=True)
           (fc2): Linear(in_features=3072, out_features=768, bias=True)
           (final_layer_norm): L

Now, let's implement the `get_model_info` function.

In [ ]:
def get_model_info(model):
    """
    Returns basic information about the loaded LLM model.

    Args:
        model: The loaded model

    Returns:
        dict: Dictionary containing model information
    """

    ### To Complete ###

### 2.2 Investigate Tokenization Behavior

Next, you'll explore how tokenizers process input text, especially when the text contains typos or other variations.

**Why is this important?**
Tokenization is the process of converting text into a sequence of tokens, which are the basic units of text that the model can process. Understanding how tokenizers handle typos or other input variations is crucial for ensuring that the model receives the correct input and for improving the model's quality and efficiency.

**Your tasks:**
1. Use the tokenizer to encode both a correctly spelled sentence and a version with typos (or other input variations).
2. Repeat this process for several different LLMs to observe any differences.

**What to think about:**
- In what ways do typos or mistakes in the input sentence affect the resulting tokenization?
- How might these tokenization differences influence the model's downstream performance?

In [ ]:
# Test sentences - one with typos and one without
correct_sentence = "The quick brown fox jumps over the lazy dog."
typo_sentence = "Teh quik brwn fox jmps ovr the lzy dog."

### To Complete ###

### 2.3 Explore LLM Module Composition

Now let's focus on systematically analyzing the internal building blocks ("modules") that make up a large language model (LLM).

**Why is this important?**
Understanding the types and counts of modules within an LLM helps you reason about its computational structure, memory footprint, and the architectural patterns that enable its capabilities. This knowledge is foundational for model comparison, optimization, and interpretability.

**Your tasks:**
1. Implement the `count_modules`function that traverses a loaded LLM and returns a count of each unique module type it contains.
2. Apply your utility to multiple models to extract and compare module statistics across several LLMs. This will help you observe architectural similarities and differences.

**What to think about:**
- What is the role of each module type in the model's computation?
- Which modules appear only once, and why might that be?
- Which modules are repeated many times, and what does this suggest about the model's design?
- Which module type is the most common, and what does this reveal about the model's dominant operations?

In [ ]:
def count_modules(model):
    """Count the number of each type of module in the model.

    Args:
        model: The PyTorch model to analyze

    Returns:
        dict: A dictionary mapping module type names to their counts
    """
    module_counts = {}

    ### To Complete ###

### 2.4 Explore LLM Parameter Distribution

This time, you'll systematically analyze the parameter distribution across different module types within a large language model (LLM).

**Why is this important?**
Understanding how parameters are distributed among module types helps you reason about the model's memory usage, computational hotspots, and architectural design choices. This insight is crucial for model optimization, interpretability, and efficient deployment.

**Your tasks:**
1. **Implement a parameter counting utility:**  
   Write a class-based utility that traverses a loaded LLM and returns a count of parameters for each unique module type it contains.
2. **Apply your utility to multiple models:**  
   Use your class to extract and compare parameter statistics across several LLMs. This will help you observe architectural similarities and differences.

**What to think about:**
- Which modules have the least parameters, and why? What does this suggest about their efficiency impact?
- Which modules have the most parameters, and why? What does this suggest about their efficiency impact?
- What is the parameter count for each module type?
- Which layer type dominates the parameter count, and what does this reveal about the model's design?

Let's implement the code to answer the questions.

In [ ]:
def count_module_parameters(model):
    """Count the number of parameters in each module type in the model.

    Args:
        model: The PyTorch model to analyze

    Returns:
        dict: A dictionary mapping module type names to their total parameter counts
    """
    module_parameters = {}

    ### To Complete ###

Now, you'll systematically analyze the precision (dtype) of parameters across different module types within a large language model (LLM).

**Why is this important?**
Understanding the precision of parameters in each module type helps you reason about the model's memory usage, numerical stability, and potential for quantization or mixed-precision optimization. This insight is crucial for efficient deployment and hardware compatibility.

**Your tasks:**
1. Implement the `get_module_precisions`function that traverses a loaded LLM and returns the precision (dtype) of parameters for each unique module type that contains parameters.
2. Apply your utility to multiple models to extract and compare parameter precision statistics across several LLMs. This will help you observe architectural similarities and differences in precision usage.

**What to think about:**
- What is the precision of the parameters in each module?
- Are there modules with mixed precision, and what might this indicate?
- How does parameter precision vary across different LLM architectures?

In [ ]:
def get_module_precisions(model):
    """
    Returns a dictionary mapping module types to their parameter precision/dtype.

    Args:
        model: PyTorch model

    Returns:
        dict: Mapping of module type names to their parameter dtype
    """
    module_dtypes = {}

    ### To Complete ###

As we saw in the previous section, the number of parameters in a model of a model can be misleading as it only looks at the number of parameters in the model, but not the precision of the parameters. We will continue with the same model, and we will explore how to estimate the memory footprint of a model based on the precision and number of parameters of each module.

**Your tasks:**
1. Implement the `estimate_memory_footprint` utility that takes both the parameter count and the precision (dtype) for each module type, and estimates the memory footprint of the model.
2. Apply your utility to multiple models to extract and compare memory usage statistics across several LLMs. This will help you observe how memory requirements differ based on architecture and precision.

**What to think about:**
- What is the estimated memory footprint of each module type, given its parameter count and precision?
- What is the total estimated memory footprint of the model?
- What are the limitations or assumptions of this estimation approach?

In [ ]:
def estimate_memory_footprint(
    module_parameters: dict, module_precisions: dict
) -> tuple[dict, float]:
    """
    Estimates the memory footprint of a model based on parameter counts and precisions.

    Args:
        module_parameters: Dict mapping module types to parameter counts
        module_precisions: Dict mapping module types to parameter dtypes

    Returns:
        dict: Memory usage in bytes for each module type
        float: Total memory usage in bytes
    """

    ### To Complete ###

Besides weights, we can also visualize the internal structure of model parameters when it comes to the initialization of the parameter weights. The initialization of the parameters is a crucial step in the training process, and it can be used to control the model's behavior. Therefore, it is important to understand how the parameters are initialized and to learn if we can find any patterns in the initialization of the parameters.

**Your tasks:**
1. Implement a parameter visualization utility, `plot_weight_heatmap_and_distribution`, which, given a model and a module name, generates:
   - A heatmap of the absolute values of the parameters.
   - A distribution plot (histogram) of the parameter values.

2. Apply your utility to multiple LLMs to extract and compare parameter visualizations across several LLMs. This will help you observe whether there are any patterns in the parameter initialization or learned weights.

**What to think about:**
- Does the parameter heatmap show any patterns or structure?
- Does the parameter distribution reveal any interesting characteristics?

In [ ]:
def plot_weight_heatmap_and_distribution(
    model, module_name: str, title: str = None
) -> None:
    """
    Plot side-by-side the heatmap of absolute weight values and the distribution of weights
    for a specific module in the model.

    Args:
        model: PyTorch model
        module_name: Name of the module to analyze
        title: Optional title for the plot
    """

    ### To Complete ###

### 2.5 Analyze LLM Activations

For the next section, you'll systematically investigate the activations produced by large language models (LLMs) in response to specific input sentences.

**Why is this important?**
Examining the activations at various layers provides insight into how information is processed and transformed throughout the model. This understanding is crucial for interpreting model behavior, diagnosing issues, and optimizing architectures for efficiency and performance.

**Your tasks:**
1. Implement an activation collection utility, `colllect_activations`, which processes a given input sentence through a loaded LLM and records the activations at each relevant layer.
2. Apply your utility to multiple models to extract and compare activation statistics across several LLMs. This will help you observe how different architectures represent and propagate information.

**What to think about:**
- What is the shape of the activations at each layer?
- How do activation patterns differ between models and layers?
- What do these differences suggest about the models' internal representations?

In [ ]:
def collect_activations(model, tokenizer, prompt: str) -> dict:
    """
    Collect activation values when processing an input prompt through the model.

    Args:
        model: PyTorch model
        tokenizer: Tokenizer
        prompt: Input text prompt

    Returns:
        Dictionary mapping layer names to their activation values
    """
    activations = {}

    ### To Complete ###

Now that we can collect the activations of the model, it's valuable to visualize the internal structure of these activations produced by LLMs in response to different inputs. Understanding the distribution and structure of these activations can provide insights into how information is processed within the model.

**Your tasks:**
1. Develop `plot_activation_heatmap_and_distribution_side_by_side` to visualize activation, which given a model, a module name, and an input prompt, generates:
   - A heatmap of the absolute values of the activations.
   - A distribution plot (histogram) of the activation values.

2. Apply your utility to multiple LLMs and input prompts to extract and compare activation visualizations across several LLMs and a variety of input sentences. This will help you observe whether there are any consistent patterns in the activations or notable differences between models or prompts.

**What to think about:**
- Do the activation heatmaps reveal any structure or recurring patterns?
- Does the distribution of activation values show any interesting characteristics or trends?

In [ ]:
def plot_activation_heatmap_and_distribution_side_by_side(
    activation_tensor, title: str = None
) -> None:
    """
    Plot a heatmap and a distribution histogram of an activation tensor side by side.

    Args:
        activation_tensor: PyTorch tensor of activations
        title: Title for the plot
    """

    ### To Complete ###

### 2.6 Investigate LLM Attention Scores

Finally, you'll systematically examine the attention scores produced by large language models (LLMs) in response to specific input sentences.

**Why is this important?**
Analyzing attention scores at various layers provides valuable insight into how the model distributes its focus across different tokens in the input. This understanding is crucial for interpreting model behavior, diagnosing issues, and optimizing architectures for efficiency and performance.

**Your tasks:**
1. Implement an attention score collection utility, `collect_attention_scores`, which processes a given input sentence through a loaded LLM and records the attention scores at each relevant layer.
2. Apply your utility to multiple models and prompts to extract and compare attention score statistics across several LLMs and input sentences. This will help you observe how different architectures and prompts influence attention patterns.

**What to think about:**
- What is the shape of the attention scores at each layer?
- How do attention patterns differ between models, layers, and input sentences?
- What do these differences suggest about the models' internal mechanisms?

In [ ]:
def collect_attention_scores(model, tokenizer, prompt: str) -> dict:
    """
    Collect attention scores from all layers when processing a prompt.

    Args:
        model: The transformer model
        tokenizer: The tokenizer to use
        prompt: Input text prompt

    Returns:
        Dictionary mapping layer names to attention score tensors
    """
    attention_scores = {}

    ### To Complete ###

Now that we can collect the attention scores from the model, it's valuable to visualize these scores to better understand how the model distributes its focus across the input sequence. Visualizing the attention patterns can reveal structural insights about how information flows within the model.

**Your tasks:**
1. Develop `plot_attention_heatmap` to visualize attention, so that, given a model, a module name, and an input prompt, generates:
   - A heatmap of the absolute values of the attention scores for a selected layer and head.

2. Apply your utility to multiple LLMs and input prompts to extract and compare attention visualizations across several LLMs and a variety of input sentences. This will help you observe whether there are any consistent patterns in the attention or notable differences between models or prompts.

**What to think about:**
- Do the attention heatmaps reveal any structure or recurring patterns?
- How do attention patterns differ between models, layers, and input sentences?

In [ ]:
def plot_attention_heatmap(attention_scores, layer_idx: int = 0, head_idx: int = 0):
    """
    Plot attention heatmap for a specific layer and attention head.

    Args:
        attention_scores: Dictionary of attention scores from collect_attention_scores()
        layer_idx: Index of the transformer layer to visualize (default: 0)
        head_idx: Index of the attention head to visualize (default: 0)
    """

    ### To Complete ###

## Conclusion: What We've Learned About LLM Architectures

In this module, we explored the inner workings of large language models (LLMs) by analyzing their architectures, parameters, and attention mechanisms. Here’s a recap of the key concepts and skills you’ve developed:

- **Model Architecture Exploration:**
  You learned how to load and inspect the structure of popular LLMs, examining their layers, embedding sizes, and attention mechanisms.

- **Parameter Analysis:**
  We broke down the parameter counts for different components, helping you understand where most of the model’s capcacity and memory load resides.

- **Attention Analysis:**
  You built utilities to extract and visualize attention scores, revealing how LLMs focus on different parts of the input. This provided insight into the interpretability and behavior of transformer models. It also hint about what can be the compute load of the attention mechanism.

- **Comparative Analysis:**
  By comparing attention patterns across models and prompts, you gained intuition about how architectural choices and data affect model behavior.

### Next Steps: Optimize your Hardware

Now that you understand the foundations of LLM architectures, the next segment will focus on **optimizing your hardware**. You’ll learn how to run LLMs on CPU and GPU, which are the main differences between the two and how to measure the performance of the LLM.

👉 **Continue to the next notebook:**
[02-run_llm_cpu_vs_gpu.ipynb on GitHub](https://github.com/PrunaAI/ai-efficiency-courses/blob/main/exercises/02-run_llm_cpu_vs_gpu.ipynb)

## ⭐ Bonus Exercise: Analyze Quantized Models

As a bonus, try applying your analysis tools to a **quantized model**. Quantization can dramatically reduce model size and inference cost—see if you can spot any differences in architecture or attention patterns!

You can find ready-to-use Pruna [quantized models on Hugging Face](https://huggingface.co/models?other=bitsandbytes&sort=trending&search=pruna).

**Your tasks:**
1. **Load a quantized model:**
   Load a quantized model and apply your analysis tools to it.
2. **Compare quantized and non-quantized models:**
   Compare the results of your analysis between the quantized and non-quantized models.

**What to think about:**
- What are the differences in the architecture of the quantized and non-quantized models?